In [40]:
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import re
from pypinyin import lazy_pinyin
from rapidfuzz import fuzz
from mutagen.mp3 import MP3
import math
from uuid import uuid4 as uuid
from scipy.optimize import linear_sum_assignment
from dotenv import load_dotenv
import subprocess
from tqdm import tqdm
load_dotenv(".env")

root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
# df = pd.read_pickle("song_embeddings_large.pkl")
# embeddings = np.vstack(df["embedding"].values).astype(np.float32)
# embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
df = pd.read_csv("songs.csv")
df.sample()

,code,type,title,lyrics,pinyin
156,W4-14,PnW,我要颂扬,我要颂扬颂扬那造眼睛的主\n因为万事他都看见\n我要颂扬颂扬那造耳朵的主\n因为万事他都听见...,wo yao song yang song yang na zao yan jing de ...


In [ ]:
def get_embedding(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )
    return response.data[0].embedding


def pinyin(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    result = " ".join(lazy_pinyin(text))
    return re.sub(r"\s+", " ", result).strip()


def log_score(x, k=0.1):
    return math.log(1 + k*x) / math.log(1 + 100*k)


def windows(tokens, size, step):
    if len(tokens) <= size:
        yield " ".join(tokens)
    else:
        for i in range(0, len(tokens) - size + 1, step):
            yield " ".join(tokens[i:i+size])


def best_window_score(query_py, lyrics_py, size=50, step=10):
    query_tokens = query_py.split()
    lyric_tokens = lyrics_py.split()
    score = max(
        fuzz.ratio(qw, lw)
        for qw in windows(query_tokens, size, step)
        for lw in windows(lyric_tokens, size, step)
    )
    return score / 100


def weighted_avg(a, b, alpha=0.5, beta=0.5):
    return (a * alpha + b * beta) / 2


def match_zoom_to_sq(d):
    files = sorted(os.listdir(f"{root}/{d}"))
    zoom_files = {f: os.path.getsize(f"{root}/{d}/{f}") for f in files if f.startswith("ZOOM")}
    other_files = {f: os.path.getsize(f"{root}/{d}/{f}") for f in files if not f.startswith("ZOOM")}
    if not zoom_files:
        return {}
    zoom_items = list(zoom_files.items())
    other_items = list(other_files.items())
    cost = np.array([
        [abs(z_size - o_size) for _, o_size in other_items]
        for _, z_size in zoom_items
    ])
    rows, cols = linear_sum_assignment(cost)
    matches = {
        zoom_items[r][0]: other_items[c][0]
        for r, c in zip(rows, cols)
    }
    return matches


def get_duration(filepath):
    duration = float(subprocess.check_output([
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        filepath,
    ]).decode().strip())
    return duration


def split_to_limit(filepath, limit=26_214_400, margin=0.90, out_dir="tmp"):
    size = os.path.getsize(filepath)
    if size <= limit:
        return [filepath]
    os.makedirs(out_dir, exist_ok=True)
    duration = get_duration(filepath)
    bitrate_kbps = 128
    chunk_seconds = max(1, int(limit * margin * 8 / (bitrate_kbps * 1000)))
    chunk_paths = []
    
    for start in range(0, math.ceil(duration), chunk_seconds):
        chunk_path = os.path.join(out_dir, f"{uuid()}.mp3")
        subprocess.run([
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-t", str(chunk_seconds),
            "-i", filepath,
            "-vn",
            "-c:a", "libmp3lame",
            "-b:a", f"{bitrate_kbps}k",
            chunk_path,
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        chunk_paths.append(chunk_path)
        print(
            f"chunk {len(chunk_paths)}: "
            f"{os.path.getsize(chunk_path):,} bytes "
            f"(limit {limit:,}) -> {chunk_path}"
        )
    return chunk_paths

In [33]:
for d in sorted(os.listdir(root), reverse=True):
    if not d.startswith("2026-"):
        continue
    for f in sorted(os.listdir(f"{root}/{d}")):
        if bool(re.search(r'[\u4e00-\u9fff]', f)):
            continue
        filepath = f"{root}/{d}/{f}"
        duration = get_duration(filepath)
        mins, secs = int(duration // 60), int(duration % 60)
        print(f"{filepath} {mins:02d}:{secs:02d}")

/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-06-14/SQ-ST388.mp3 00:32
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-04-19/SQ-ST323.mp3 01:47
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-04-19/ZOOM0326.mp3 01:43
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-07/SQ-ST262.mp3 00:10
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-02-28/SQ-ST251.mp3 00:11
/mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-02-28/ZOOM0248.mp3 00:11


In [19]:
print(split_to_limit(f"{root}/2026-05-08 (祷告会)/SQ-ST346.mp3"))

chunk 1: 23,585,062 bytes (limit 26,214,400) -> tmp/b76a32bb-97fc-499c-9cf6-2ce0c8f9af7f.mp3
chunk 2: 23,585,062 bytes (limit 26,214,400) -> tmp/35b7d494-22fc-4bff-91c0-113629c84c61.mp3
chunk 3: 23,585,062 bytes (limit 26,214,400) -> tmp/eaaceb3c-0d40-4fd6-b087-24c4e3342857.mp3
chunk 4: 23,344,318 bytes (limit 26,214,400) -> tmp/9a0d72a0-bbe4-46e5-a8ac-4e17bfcf9cba.mp3
['tmp/b76a32bb-97fc-499c-9cf6-2ce0c8f9af7f.mp3', 'tmp/35b7d494-22fc-4bff-91c0-113629c84c61.mp3', 'tmp/eaaceb3c-0d40-4fd6-b087-24c4e3342857.mp3', 'tmp/9a0d72a0-bbe4-46e5-a8ac-4e17bfcf9cba.mp3']


In [34]:
# filepath = f"{root}/2026-01-24/SQ-ST196.mp3"
lyrics = ""
for filepath in tqdm([f"{root}/2026-04-19/SQ-ST323.mp3"]):
    audio_file = open(filepath, "rb")
    transcription = client.audio.transcriptions.create(
        # model="gpt-4o-transcribe", 
        model="whisper-1", 
        file=audio_file,
        language="zh",
    )
    lyrics += transcription.text
lyrics

100%|██████████| 1/1 [00:04<00:00,  4.92s/it]


'和華世家表明一心 忍住坦然無懼 從今以後一生一世 皆歸北地之主 和華世家表明一心 虔誠頌足榮名 九族雄辱九族光榮 在你克謙山林 和華世家表明一心 勇征楚路而行 萬佛世家感受羞辱 向前行走天長 如今為你和華世家 表明一歸中國 但願今生世家之老 將來更愛華光'

In [35]:
titles = {}
title_to_last_chunk_idx = {}

query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

chunk_size = 120
for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
    chunk = query_lyrics[start:start + chunk_size]
    if len(chunk) < 50:
        continue
    # query_embedding = get_embedding(chunk)
    # query_embedding /= np.linalg.norm(query_embedding)
    # scores = embeddings @ query_embedding
    # alpha, beta = (0.1, 0.9) if len(chunk) < chunk_size / 2 else (0.3, 0.7)
    # scores = [weighted_avg(
    #     score, best_window_score(pinyin(chunk), df.pinyin[i], size=len(chunk)//2, step=5), alpha, beta,
    # ) for i, score in enumerate(scores)]
    query_py = pinyin(chunk)
    scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
    best_idx = np.argmax(scores)
    best_title = df.iloc[best_idx]["title"]
    best_score = scores[best_idx]
    print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
    if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
        titles[best_title] = max(titles[best_title], best_score) * 1.2
    else:
        titles[best_title] = best_score
    title_to_last_chunk_idx[best_title] = i

print(f"{titles=}")
final_titles = [title for title, score in titles.items() if score > 0.7]
print(f"Songs: {'_'.join(final_titles)}")

[0:120] best_title='求主使我近十架', best_score=0.56
titles={'求主使我近十架': 0.56}
Songs: 


In [12]:
print(" ".join(df.loc[df.title == "祢爱永不变"].lyrics.to_list()))

祢流出宝血，洗净我污秽，将我的生命赎回。祢为了我的罪，牺牲永不悔，显明祢极大恩惠。我深深体会，祢爱的宝贵，献上自己永追随。或伤心或气馁，或生离或死别，愿刚强壮胆永远不后退。哦，祢爱永不变，从今直到永远，深深浇灌我心田。或天旋或地转，经沧海历桑田，都不能叫我与祢爱隔绝。


In [17]:
chunk = query_lyrics[2640:2760]
print("Query:", chunk)
query_pinyin = pinyin(chunk)
query_embedding = get_embedding(chunk)
query_embedding /= np.linalg.norm(query_embedding)
scores = embeddings @ query_embedding
for t in ["宁静谷"]:
    inds = df.loc[df.title == t].index
    for idx in inds:
        print(f"[{idx}] {t}: {df.pinyin[idx]}")
        score = scores[idx]
        fuzz_score = best_window_score(query_pinyin, df.pinyin[idx], size=100, step=3)
        print(f"{score=}, {fuzz_score=}, {weighted_avg(score, fuzz_score, 0.3, 0.7)}")

Query:  我学会了信靠他 依靠他 有一次当我 向一位朋友 倾诉我的挣扎时 他推荐我 他推荐给我一首 藏民之群的歌 叫《宁静谷》 歌词中写道 生活中的仓促 生命里的难处 只愿向他来倾诉 平安祝福在这谷 我觉得这首歌 正好讲述了 那段时期 上帝如何 把
[77] 宁静谷: zai wo xin ling shen chu you yi zuo ning jing gu wo he wo qin ai de zhu zai qi zhong an ran man bu sheng huo zhong de cang cu sheng ming li de nan chu zhi yuan xiang ta lai qing su ping an zhu fu zai zhe gu wo yu wo zhu xiang yue zhi chu chang yang zhe fen ning jing an xiang jiu xiang shi zai tian tang wo yu wo zhu xiang yue zhi chu zhu ling wo guo si yin you gu shi wo xi le zou ren sheng lu
score=0.5467158003484595, fuzz_score=0.5852417302798982, 0.2868419756502333


In [ ]:
def get_titles(filepath):
    print(f"Processing {filepath}")

    cropped_paths = split_to_limit(filepath)
    lyrics = ""
    for filepath in tqdm(cropped_paths):
        audio_file = open(filepath, "rb")
        transcription = client.audio.transcriptions.create(
            # model="gpt-4o-transcribe", 
            model="whisper-1", 
            file=audio_file,
            language="zh",
        )
        lyrics += transcription.text

    if not lyrics:
        return []

    titles = {}
    title_to_last_chunk_idx = {}

    query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

    chunk_size = 120
    for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
        chunk = query_lyrics[start:start + chunk_size]
        if len(chunk) < 50:
            continue
        query_py = pinyin(chunk)
        scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
        best_idx = np.argmax(scores)
        best_title = df.iloc[best_idx]["title"]
        best_score = scores[best_idx]
        print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
        if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
            titles[best_title] = max(titles[best_title], best_score) * 1.2
        else:
            titles[best_title] = best_score
        title_to_last_chunk_idx[best_title] = i

    print(f"{titles=}")
    duration = get_duration(filepath)
    if duration > 3 * 60:
        final_titles = [title for title, score in titles.items() if score > 0.7]
    else:
        best_title = max(titles, key=titles.get)
        final_titles = [best_title] if titles[best_title] > 0.7 else []
    return final_titles

In [40]:
final_titles = get_titles(f"{root}/2026-03-29/SQ-ST303.mp3")
print(f"Songs: {'_'.join(final_titles)}")

Processing /mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-03-29/SQ-ST303.mp3
chunk 1: 23,593,840 bytes (limit 26,214,400) → tmp/d36b98af-e251-4a97-9e32-f229340e3413.mp3
chunk 2: 17,264,266 bytes (limit 26,214,400) → tmp/8728ed6f-6e0e-4e42-be40-472756109f89.mp3
[0:120] best_title='爱使我们勇敢+我们爱', best_score=0.5943600867678959
[120:240] best_title='我们爱戴的王', best_score=0.6477732793522267
[240:360] best_title='我们爱戴的王', best_score=0.6652360515021459
[360:480] best_title='生命的凯歌', best_score=0.5708661417322834
[480:600] best_title='我们爱戴的王', best_score=0.6117136659436009
[600:720] best_title='生命之道极奇', best_score=0.5978947368421053
[720:840] best_title='生命之道极奇', best_score=0.5978947368421053
[840:960] best_title='生命之道极奇', best_score=0.5978947368421053
[960:1080] best_title='生命之道极奇', best_score=0.5889830508474576
[1080:1200] best_title='神的爱', best_score=0.5630252100840336
[1200:1320] best_title='无价至宝', best_score=0.9443207126948775
[1320:1440] best_title='让我得见你的荣耀', best_score=0.5634408602150

['我的盼望在于祢']

In [ ]:
for d in sorted(os.listdir(root)):
    if not d.startswith("2026-05"):
        continue
    files = sorted(os.listdir(f"{root}/{d}"))
    if any(bool(re.search(r'[\u4e00-\u9fff]', f)) for f in files):  # already renamed
        continue
    zoom_to_sq = match_zoom_to_sq(d)
    zoom_files = [f for f in files if f.startswith("ZOOM")]
    sq_files = [f for f in files if not f.startswith("ZOOM")]
    file_to_titles = {f: get_titles(f"{root}/{d}/{f}") for f in sq_files}
    for zoom_file in zoom_files:
        sq_file = zoom_to_sq.get(zoom_file)
        sq_titles = file_to_titles.get(sq_file, [])
        titles = get_titles(f"{root}/{d}/{zoom_file}") or sq_titles
        if any(t in sq_titles for t in titles):
            titles = sq_titles
        file_to_titles[zoom_file] = titles
    for f, title in file_to_titles.items():
        filename, ext = os.path.splitext(f)
        new_filepath = f"{filename}_{'_'.join(title)}{ext}" if title else f"{filename}{ext}"
        print(f"{f}→{new_filepath}")
        if f != new_filepath:
            os.rename(f"{root}/{d}/{f}", f"{root}/{d}/{new_filepath}")
    !sudo -u www-data php /var/www/html/nextcloud_sacm/occ files:scan --path "sacm.av/files/Recordings/{d}"

Processing /mnt/NextcloudSacmData/sacm.av/files/Recordings/2026-05-07/SQ-ST345.mp3
